# Correcciones Examen Práctico 1

En este notebook se presentan las correcciones de los ejercicios prácticos del examen.

Para cada corrección se incluyen los cuatro puntos solicitados:

- ¿Cuál fue el error?
- ¿Cuál es la corrección?
- ¿Por qué se cometió el error?
- ¿Cómo se puede evitar este error en el futuro?

**Nota:** El ejercicio 1 requiere que `ex1_01.xlsx` esté en la misma carpeta para ejecutar sus celdas.


# Ejercicio 1

Se crea un contrato de compra donde se quiere que la acción sea estable. La persona con la que estás haciendo el contrato accede a comprar el 50% de las acciones de tu empresa si la volatilidad móvil con n=20 (anualizada) no excede el 20%. El archivo `ex1_01.xlsx` contiene dos rutas. ¿En cuál de las dos situaciones se activa el contrato?


## Corrección 1 — Volatilidad móvil

**¿Cuál fue el error?**  
En el examen coloqué `vol_1.append(desviacion_1)` y `vol_2.append(desviacion_2)` fuera del ciclo `for`. Por esto, las listas guardaron únicamente la desviación estándar de la última ventana de 20 rendimientos y no todas las volatilidades móviles.

**¿Cuál es la corrección?**  
Los `append()` deben estar dentro del `for` para guardar una volatilidad por cada ventana de 20 rendimientos.

**¿Por qué se cometió el error?**  
Confundí el cálculo de una sola desviación estándar con el almacenamiento de todas las desviaciones de las ventanas móviles. El `for` sí recorría todas las ventanas, pero al terminar solamente se guardaban los últimos valores calculados.

**¿Cómo se puede evitar este error en el futuro?**  
Después de crear una lista dentro de un ciclo, verificaré que el número de resultados sea igual al número esperado de ventanas con `len(vol_1) == len(rendimiento_A)-n+1`. Así puedo detectar inmediatamente si el ciclo está almacenando un resultado en cada iteración.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel('ex1_01.xlsx')
df.head()

In [ ]:
df['Rendimiento_A'] = np.log(df['Ruta A']/df['Ruta A'].shift(1))
df['Rendimiento_B'] = np.log(df['Ruta B']/df['Ruta B'].shift(1))

rendimiento_A = df['Rendimiento_A'].dropna()
rendimiento_B = df['Rendimiento_B'].dropna()

In [ ]:
n = 20
vol_1 = []
vol_2 = []

for i in range(len(rendimiento_A)-n+1):
    valores_1 = rendimiento_A.iloc[i:i+n]
    valores_2 = rendimiento_B.iloc[i:i+n]
    desviacion_1 = np.std(valores_1, ddof=1)
    desviacion_2 = np.std(valores_2, ddof=1)
    vol_1.append(desviacion_1)
    vol_2.append(desviacion_2)

vol_1 = np.array(vol_1)
vol_2 = np.array(vol_2)

print('Ventanas esperadas:', len(rendimiento_A)-n+1)
print('Ventanas guardadas A:', len(vol_1))
print('Ventanas guardadas B:', len(vol_2))

In [ ]:
vol_1_anual = vol_1*np.sqrt(252)
vol_2_anual = vol_2*np.sqrt(252)

limite = 0.20
max_A = np.max(vol_1_anual)
max_B = np.max(vol_2_anual)

print('Máxima volatilidad anual de A:', max_A)
print('Máxima volatilidad anual de B:', max_B)

cumple_A = np.all(vol_1_anual <= limite)
cumple_B = np.all(vol_2_anual <= limite)

print('Ruta A cumple:', cumple_A)
print('Ruta B cumple:', cumple_B)

### Conclusión

El contrato se activa únicamente en la ruta cuya volatilidad móvil anualizada se mantenga en o por debajo de 20% durante todas las ventanas analizadas. La decisión se toma comparando el máximo de cada serie de volatilidades móviles contra `0.20`.


# Ejercicio 2

Utilizando `S0=100`, `r=0.05`, `sigma=0.20`, `T=1`, `n=252` y un GBM:

- A partir del día 30 la tasa aumenta a 8%.
- A partir del día 50 la volatilidad aumenta a 40% durante 20 días y después regresa a 20%.
- En el día 120 ocurre un anuncio que históricamente produce reacciones fuertemente positivas e inmediatas durante una semana.
- Se debe realizar una simulación Monte Carlo con `M=10000`.
- Se deben mostrar al menos 20 trayectorias y marcar los cambios abruptos.


## Corrección 2 — No utilicé M=10000

**¿Cuál fue el error?**  
En el examen definí `M=10000`, pero el código solamente generó una trayectoria de precios. Por lo tanto, no realicé la simulación Monte Carlo solicitada con 10,000 trayectorias.

**¿Cuál es la corrección?**  
Crear una matriz de tamaño `(M, n+1)` para almacenar las 10,000 trayectorias y actualizar simultáneamente los precios de todas las simulaciones en cada día.

**¿Por qué se cometió el error?**  
Tomé como base el código de un GBM de una sola trayectoria y no adapté su estructura al parámetro `M`.

**¿Cómo se puede evitar este error en el futuro?**  
Antes de programar, revisaré que cada parámetro del enunciado aparezca realmente en la lógica del código. Si existe `M`, comprobaré la dimensión de la matriz con `precios.shape` y verificaré que sea `(10000, 253)`.


## Corrección 3 — No incorporé los cambios de tasa y volatilidad

**¿Cuál fue el error?**  
Utilicé `r=0.05` y `sigma=0.20` durante toda la trayectoria, aunque el problema indicaba que la tasa debía cambiar a 8% desde el día 30 y que la volatilidad debía subir a 40% durante 20 días desde el día 50.

**¿Cuál es la corrección?**  
Dentro del ciclo diario se deben definir `r_dia` y `sigma_dia` mediante condiciones. Desde el día 30 se usa `r_dia=0.08`. Entre los días 50 y 69 se usa `sigma_dia=0.40`; fuera de ese intervalo se usa `sigma_dia=0.20`.

**¿Por qué se cometió el error?**  
Apliqué directamente la fórmula de GBM con parámetros constantes y no separé el problema en eventos que modificaban los parámetros según el día.

**¿Cómo se puede evitar este error en el futuro?**  
Antes de escribir el GBM, convertiré el enunciado en una línea de tiempo y probaré los valores de los parámetros en días clave, por ejemplo 29, 30, 49, 50, 69 y 70. Esto permite comprobar que cada cambio inicia y termina en el día correcto.


## Corrección 4 — No incorporé la reacción positiva del día 120

**¿Cuál fue el error?**  
No incorporé el efecto del anuncio programado para el día 120.

**¿Cuál es la corrección?**  
El enunciado no proporciona una magnitud numérica para la reacción positiva. Para no inventar un porcentaje de rendimiento, se modela la información cualitativa haciendo que durante una semana bursátil (5 días, del 120 al 124) los choques aleatorios sean positivos mediante `np.abs(Z)`. De esta manera se representa una reacción positiva temporal y después se regresa al proceso normal.

**¿Por qué se cometió el error?**  
No supe cómo convertir una condición cualitativa como “reacciones fuertemente positivas e inmediatas” en una modificación del modelo porque el problema no daba una magnitud específica.

**¿Cómo se puede evitar este error en el futuro?**  
Cuando un enunciado incluya información cualitativa sin magnitud, identificaré explícitamente qué parte es un supuesto de modelación y lo escribiré en Markdown antes de programarlo. También evitaré asignar porcentajes que no hayan sido proporcionados sin explicar el supuesto.


## Corrección 5 — Gráfica incompleta

**¿Cuál fue el error?**  
En el examen grafiqué una sola trayectoria y no marqué los días donde cambiaban las condiciones del modelo.

**¿Cuál es la corrección?**  
Graficar al menos 20 de las 10,000 trayectorias y agregar líneas verticales en los días 30, 50, 70, 120 y 125 para identificar el cambio de tasa, el inicio y fin de la volatilidad alta y el inicio y fin del periodo de reacción positiva.

**¿Por qué se cometió el error?**  
La gráfica se construyó a partir del GBM básico de una trayectoria y no verifiqué todos los elementos solicitados en la sección “Entrega”.

**¿Cómo se puede evitar este error en el futuro?**  
Antes de entregar, convertiré los requisitos de salida en un checklist: número de simulaciones, número de trayectorias mostradas, marcadores, títulos, ejes y Markdown. No consideraré terminado el ejercicio hasta comprobar cada elemento.


In [ ]:
S0 = 100
r = 0.05
sigma = 0.20
T = 1
n = 252
M = 10000

dt = T/n
np.random.seed(32)

precios = np.zeros((M, n+1))
precios[:, 0] = S0

for t in range(1, n+1):
    if t >= 30:
        r_dia = 0.08
    else:
        r_dia = 0.05

    if 50 <= t < 70:
        sigma_dia = 0.40
    else:
        sigma_dia = 0.20

    Z = np.random.normal(0, 1, size=M)

    # Supuesto: una semana bursátil = 5 días
    if 120 <= t < 125:
        Z = np.abs(Z)

    precios[:, t] = precios[:, t-1]*np.exp(
        (r_dia-0.5*sigma_dia**2)*dt
        + sigma_dia*np.sqrt(dt)*Z
    )

print('Dimensión de la simulación:', precios.shape)

In [ ]:
plt.figure(figsize=(12,7))

for i in range(20):
    plt.plot(precios[i], alpha=0.6)

plt.axvline(x=30, linestyle='--', label='Día 30: r = 8%')
plt.axvline(x=50, linestyle='--', label='Día 50: sigma = 40%')
plt.axvline(x=70, linestyle='--', label='Día 70: sigma = 20%')
plt.axvline(x=120, linestyle='--', label='Día 120: anuncio')
plt.axvline(x=125, linestyle='--', label='Día 125: fin reacción positiva')

plt.title('20 trayectorias simuladas con cambios en el GBM')
plt.xlabel('Días')
plt.ylabel('Precio')
plt.legend()
plt.show()

### Justificación de las modificaciones

La simulación se realiza día por día porque los parámetros no son constantes durante todo el año. La tasa cambia de 5% a 8% a partir del día 30. La volatilidad aumenta de 20% a 40% durante exactamente 20 días, del 50 al 69, y vuelve a 20% desde el día 70.

Para el anuncio del día 120, el enunciado indica una reacción positiva durante una semana pero no proporciona una magnitud. Se considera una semana bursátil de 5 días y se utilizan choques aleatorios positivos durante ese periodo. Este es un supuesto de modelación explícito y no un valor proporcionado por el enunciado.

La matriz contiene 10,000 trayectorias y 253 columnas: el precio inicial más los 252 días simulados. En la gráfica se muestran 20 trayectorias y se señalan los días donde cambian las condiciones.
